# CKIN2 — prédiction expérimentale

Prévoir l'indice de prix global et ceux des caisses, skins, souvenirs, stickers, charms et agents à un mois. Ce prototype travaille avec les données disponibles ; il ne promet pas une précision suffisante pour décider d'une transaction.

## 1. Données et paramètres

Les prix sont ceux des séries à panier constant du notebook global. Les acquisitions et dates estimées sont exclues. Les mois calendaires terminés sont échantillonnés à leur clôture, sans interpolation. Les dates affichées désignent le jour de clôture. Le mois en cours est toujours exclu. Les absences de prix au-delà de 45 jours restent manquantes.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# Override only the data directory for reproducible synthetic validation.
DATA_ROOT = Path(os.environ.get("CKIN2_DATA_ROOT", str(PROJECT_ROOT)))
from portfolio_data import load_holdings, load_prices
from prediction import experiment

AS_OF = pd.Timestamp.now()
MIN_TRAIN = 12
MAX_PRICE_AGE_DAYS = 45
holdings, imported = load_holdings(DATA_ROOT)
prices = load_prices(DATA_ROOT, holdings)
prediction = experiment(holdings, prices, AS_OF, MIN_TRAIN, MAX_PRICE_AGE_DAYS)
display(pd.DataFrame([prediction["diagnostic"]]))


## 2. Audit de l'historique

Consulter le nombre de mois valides, de trous, de fenêtres évaluables et de paires d'entraînement. Sans base commune ou historique suffisant, le modèle reste indisponible. Douze paires sont un seuil technique de démarrage, pas une preuve de qualité statistique.

In [ ]:
display(prediction["audit"])
monthly = prediction["monthly"]
display(monthly)
if not monthly.empty:
    px.line(monthly, x="date", y="index_100", color="category", markers=True, hover_data=["age_days"], title="Indices mensuels disponibles pour l'expérience").show()
else:
    print("Aucune série comparable : compléter la couverture de marché.")


## 3. Validation chronologique

À chaque fin de mois, prévoir uniquement le mois suivant. Prévisions simples : dernier indice connu et prolongement de la variation du dernier mois. Premier modèle : StandardScaler puis Ridge (alpha=1, solveur SVD), ajustés à nouveau uniquement sur les paires dont la cible est déjà connue à l'origine. Variables : dernier indice, dernière variation et moyenne des trois derniers mois. Aucun mélange aléatoire, aucune optimisation sur le futur. Les valeurs prévues sont bornées à zéro.

In [ ]:
backtests = prediction["backtests"]
display(backtests)
display(prediction["scores"])
if not backtests.empty:
    global_tests = backtests.loc[backtests.category.eq("global")]
    if not global_tests.empty:
        px.line(global_tests, x="target_date", y=["actual", "persistence", "trend", "ridge"], markers=True, title="Indice global : observations et prévisions hors échantillon").show()
else:
    print("Au moins trois mois consécutifs valides sont nécessaires pour comparer les deux prévisions simples.")


## 4. Comparaison des erreurs

MAE et RMSE sont exprimées en points d'indice et restent définies pour des cibles nulles. Le tableau baseline_common compare les deux prévisions simples aux mêmes dates ; model_common compare les trois méthodes uniquement aux dates où Ridge était entraînable. Afficher les effectifs et dates, et ne pas comparer des scores provenant de fenêtres différentes.

In [ ]:
metrics = prediction["scores"]
if not metrics.empty:
    px.bar(metrics, x="category", y="MAE", color="method", facet_col="sample", barmode="group", hover_data=["windows", "RMSE", "first_target", "last_target"], title="Erreurs hors échantillon sur dates communes").show()
else:
    print("Pas assez de fenêtres hors échantillon pour mesurer une erreur.")


## 5. Prochaine prévision disponible

Ces prévisions partent du dernier mois complet valide disponible pour chaque série. Vérifier la date d'origine : si les derniers mois sont manquants, aucune nouvelle prévision n'est proposée. Aucun intervalle de confiance arbitraire n'est affiché. Le prix global à panier constant ne prédit pas les prochains ajouts ou ventes de ton inventaire ; une projection en EUR demanderait des hypothèses de quantités fixes explicites.

In [ ]:
display(prediction["future"])
if prediction["future"].empty:
    print("Prévision indisponible : au moins deux mois récents consécutifs valides sont nécessaires.")


## 6. Limites et suite

Le panier actuel introduit un biais de sélection rétrospectif : l'expérience n'est pas un backtest d'une stratégie investissable historique. Les prix affichés ne mesurent ni liquidité, ni frais, ni prix exécutés. Le modèle peut être moins bon que le dernier prix connu. Conserver les prévisions simples comme référence et accumuler des relevés cohérents avant d'enrichir les modèles. Revenir à `global_book.ipynb` pour analyser les valeurs et contributions.

Référence méthodologique : [scikit-learn — prévenir les fuites de données](https://scikit-learn.org/stable/common_pitfalls.html).